## Cell 1 — Install

In [0]:
%pip install langgraph langchain langchain-openai pydantic typing_extensions

## Cell 2 — Imports + Config + LLM

In [0]:
import os
from typing import Literal, List, Dict, TypedDict, Optional, Any
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

CATALOG = "deepcatalog"
SCHEMA  = "delta_tables"

os.environ["OPENAI_API_KEY"] = dbutils.secrets.get(scope="llm-scope", key="openai-key")

llm = ChatOpenAI(model="gpt-5-mini", temperature=0)

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

print("✅ Setup done")

## Cell 3 — Pydantic Models

In [0]:
class IntentFilter(BaseModel):
    column:   str
    operator: str
    value:    str

class IntentResponse(BaseModel):
    intent_type: Literal["count", "select", "aggregate", "other"] = "select"
    targets:     List[str] = Field(default_factory=list)
    filters:     List[IntentFilter] = Field(default_factory=list)
    group_by:    List[str] = Field(default_factory=list)
    complexity:  Literal["low", "medium", "high"] = "low"

class SchemaMapResponse(BaseModel):
    tables:    List[str] = Field(default_factory=list)
    join_keys: List[List[str]] = Field(default_factory=list)
    columns:   Dict[str, List[str]] = Field(default_factory=dict)

print("✅ Models defined")

## Cell 4 — Synthea Schema Reference

In [0]:
SYNTHEA_SCHEMA = {
    "tables": {
        "patients":    ["Id", "BIRTHDATE", "DEATHDATE", "GENDER", "RACE", "CITY", "STATE"],
        "conditions":  ["START", "STOP", "PATIENT", "ENCOUNTER", "CODE", "DESCRIPTION"],
        "medications": ["START", "STOP", "PATIENT", "ENCOUNTER", "CODE", "DESCRIPTION", "REASONCODE"],
        "observations":["DATE", "PATIENT", "ENCOUNTER", "CODE", "DESCRIPTION", "VALUE", "UNITS", "TYPE"],
        "encounters":  ["Id", "START", "STOP", "PATIENT", "CODE", "DESCRIPTION", "REASONCODE"],
        "procedures":  ["DATE", "PATIENT", "ENCOUNTER", "CODE", "DESCRIPTION"]
    },
    "join_keys": {
        "conditions":  ["patients.Id", "conditions.PATIENT"],
        "medications": ["patients.Id", "medications.PATIENT"],
        "observations":["patients.Id", "observations.PATIENT"],
        "encounters":  ["patients.Id", "encounters.PATIENT"],
        "procedures":  ["patients.Id", "procedures.PATIENT"]
    }
}

SAMPLED_VALUES = {
    "patients.GENDER": ["M", "F"],
    "patients.RACE":   ["white", "black", "asian", "hispanic", "native"],
    "patients.STATE":  ["Massachusetts"],
    "conditions.DESCRIPTION":  ["use LIKE '%<actual_keyword>%' with the keyword from the user query"],
    "medications.DESCRIPTION": ["use LIKE '%<actual_keyword>%' with the keyword from the user query"],
    "observations.TYPE":       ["numeric", "text"]
}

print("✅ Schema + sampled values ready")

## Cell 5 — Analyzer Agent

In [0]:
ANALYZER_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a healthcare data query analyzer working with Synthea data.
Parse the user query and return a JSON object with these exact fields:
- intent_type: one of count, select, aggregate, other
- targets: list of ALL entity types involved. Always include related entities
  (e.g., if query mentions a condition, include both "patients" and "conditions")
- filters: list of objects with column, operator, value
- group_by: list of grouping fields if any
- complexity: low, medium, or high

Return ONLY valid JSON. No explanation, no markdown, no backticks."""),
    ("human", "{query}")
])

def analyzer_node(state: dict) -> dict:
    try:
        chain    = ANALYZER_PROMPT | llm
        response = chain.invoke({"query": state["query"]})
        intent   = IntentResponse.model_validate_json(response.content)
        return {**state, "intent": intent.model_dump()}
    except Exception as e:
        return {
            **state,
            "intent": None,
            "errors": state.get("errors", []) + [f"Analyzer error: {str(e)}"]
        }

print("✅ Analyzer Agent ready")

## Cell 6 — Schema Mapper Agent

In [0]:
SCHEMA_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a healthcare database schema expert working with Synthea data.
Given the user intent, SELECT ONLY the relevant tables and columns needed to answer the query.
Do NOT return the entire schema.

Return a JSON object with:
- tables: list of ONLY the table names needed for this specific query
- join_keys: list of [left_key, right_key] pairs needed (only if joining tables)
- columns: dict of table -> list of ONLY the columns needed for this specific query

Available schema for reference:
{schema_reference}

Return ONLY valid JSON. No explanation, no markdown, no backticks."""),
    ("human", "Intent: {intent}")
])

def schema_mapper_node(state: dict) -> dict:
    try:
        chain    = SCHEMA_PROMPT | llm
        response = chain.invoke({
            "intent":           str(state["intent"]),
            "schema_reference": str(SYNTHEA_SCHEMA)
        })
        schema = SchemaMapResponse.model_validate_json(response.content)
        return {**state, "schema": schema.model_dump()}
    except Exception as e:
        return {
            **state,
            "schema": None,
            "errors": state.get("errors", []) + [f"Schema mapper error: {str(e)}"]
        }

print("✅ Schema Mapper Agent ready")

## Cell 7 — SQL Generator Agent

In [0]:
SQL_GEN_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a Spark SQL expert working with Synthea healthcare data on Azure Databricks.
Generate a valid Spark SQL query based on the intent and schema provided.

Rules:
- Always use fully qualified table names: deepcatalog.delta_tables.tablename
- Use table aliases: p=patients, c=conditions, m=medications, o=observations, e=encounters
- Use ONLY the actual sampled column values provided for filters (not assumed values)
- For partial text matches use LOWER(column) LIKE LOWER('%keyword%')
- Always qualify column names with table alias
- Return ONLY the SQL query. No explanation, no markdown, no backticks."""),
    ("human", """Intent: {intent}
Schema: {schema}
Sampled column values: {sampled_values}
Previous errors (if any): {errors}""")
])

def sql_generator_node(state: dict) -> dict:
    try:
        schema = state["schema"]

        # Read from hardcoded values — no Spark jobs
        sampled = {
            k: v for k, v in SAMPLED_VALUES.items()
            if any(k.startswith(table) for table in schema.get("tables", []))
        }

        chain    = SQL_GEN_PROMPT | llm
        response = chain.invoke({
            "intent":         str(state["intent"]),
            "schema":         str(schema),
            "sampled_values": str(sampled),
            "errors":         state.get("errors", [])
        })

        return {**state, "sql": response.content.strip()}

    except Exception as e:
        return {
            **state,
            "sql":    None,
            "errors": state.get("errors", []) + [f"SQL generator error: {str(e)}"]
        }

print("✅ SQL Generator Agent ready")

## Cell 8 — Executor

In [0]:
import re

DESTRUCTIVE = re.compile(
    r'\b(DROP|DELETE|TRUNCATE|INSERT|UPDATE|ALTER|CREATE|REPLACE)\b',
    re.IGNORECASE
)

def executor_node(state: dict) -> dict:
    sql = state.get("sql")

    if not sql:
        return {
            **state,
            "errors": state.get("errors", []) + ["Executor error: no SQL to execute"]
        }

    if DESTRUCTIVE.search(sql):
        return {
            **state,
            "result": None,
            "errors": state.get("errors", []) + ["Blocked: destructive SQL detected"]
        }

    try:
        result_df = spark.sql(sql)
        result    = result_df.collect()
        return {**state, "result": result, "result_df": result_df}
    except Exception as e:
        return {
            **state,
            "result": None,
            "errors": state.get("errors", []) + [f"Execution error: {str(e)}"]
        }

print("✅ Executor ready")

## Cell 9 — run_query()

In [0]:
SUMMARY_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful healthcare data assistant. Given a question and its result, write a clear one-sentence natural language answer."),
    ("human", """Question: {query}
SQL: {sql}
Result: {result}""")
])

def run_query(question: str):
    if DESTRUCTIVE.search(question):
        print("⛔ Access Denied: You do not have authority to run destructive commands.")
        return None

    state = {
        "query":  question,
        "intent": None,
        "schema": None,
        "sql":    None,
        "result": None,
        "errors": []
    }

    print(f"Query: {question}")
    print("-" * 50)

    state = analyzer_node(state)
    # print("\nAnalyser Agent:")
    # print(f"Intent : {state['intent']}")

    state = schema_mapper_node(state)
    # print("\nSchema Mapper Agent:")
    # print(f"Schema : {state['schema']}")

    state = sql_generator_node(state)
    # print("\nSQL Generator Agent:")
    # print(f"SQL    :\n{state['sql']}")

    state = executor_node(state)

    print("-" * 50)
    if state.get("errors"):
        print(f"❌ Errors: {state['errors']}")
    elif state.get("result") is not None:
        print("✅ Result:")
        state["result_df"].show()

        # Natural language summary
        summary = (SUMMARY_PROMPT | llm).invoke({
            "query":  question,
            "sql":    state["sql"],
            "result": str(state["result"][:5])
        }).content.strip()
        print(f"\n💬 {summary}\n")

    return state

## Cell 10 — Test

In [0]:
run_query("show how many patients are male and how many are female")